In [1]:
import numpy as np

from causal_gym.envs import MDPExample

In [2]:
GAMMA = .9
TEST_EPI = 1000

### Evaluating behavioral policy

In [3]:
import multiprocess as mp
from multiprocess import Process

def worker(policy=None):
    env = MDPExample(max_step=1000)
    s, _ = env.reset()
    done = False
    reward = 0
    while not done:
        if policy is None:
            x, s, y, _, done, _ = env.see()
        else:
            s, y, _, done, _ = env.do(policy(s))
        reward = reward * GAMMA + y
    results.append(reward)


manager = mp.Manager()
results = manager.list()
pcs = [Process(target=worker, kwargs={'policy': None}) for i in range(TEST_EPI)]
for i in range(TEST_EPI):
    pcs[i].start()
for i in range(TEST_EPI):
    pcs[i].join()
print(f'Behavioral policy avg discounted return is {sum(results)/len(results)}')

Behavioral policy avg discounted return is 0.9972473157666119


### Evaluating Optimal Atomic Policy

In [4]:
policy = lambda s: s
manager = mp.Manager()
results = manager.list()
pcs = [Process(target=worker, kwargs={'policy': policy}) for i in range(TEST_EPI)]
for i in range(TEST_EPI):
    pcs[i].start()
for i in range(TEST_EPI):
    pcs[i].join()
print(f'Atomic interventional policy avg discounted return is {sum(results)/len(results)}')

Atomic interventional policy avg discounted return is 8.242596815674684
